# 📘 환경변수와 설정 파일

프로그램의 **설정을 외부에서 관리**하는 방법을 배웁니다.
환경변수, .env 파일, INI 설정 파일을 다룹니다.

**학습 목표:**
- os.environ으로 환경변수 읽기/쓰기
- dotenv(.env) 파일 다루기
- configparser로 INI 설정 파일 관리

## 1. 환경변수

환경변수는 프로그램 외부에서 설정값을 전달하는 표준 방법입니다.
API 키, 포트 번호 등 민감한 정보를 코드에 하드코딩하지 않을 때 유용합니다.

In [ ]:
import os

# ┌─────────────────────────────────────────┐
# │  환경변수 접근법                           │
# │  os.environ["VAR"]  → 없으면 KeyError       │
# │  os.getenv("VAR")  → 없으면 None 반환       │
# │  os.getenv("VAR", 기본값) → 기본값 반환      │
# └─────────────────────────────────────────┘

# 환경변수 읽기
home = os.getenv("HOME") or os.getenv("USERPROFILE")
print(f"홈 디렉토리: {home}")

path_var = os.getenv("PATH", "/usr/bin")
print(f"PATH 길이: {len(path_var)} 문자")

# 환경변수 설정 (현재 프로세스에만 적용)
os.environ["MY_APP_DEBUG"] = "true"
os.environ["MY_APP_PORT"] = "8080"

print(f"디버그: {os.getenv('MY_APP_DEBUG')}")
print(f"포트: {os.getenv('MY_APP_PORT')}")

# 환경변수 존재 확인
if "MY_APP_DEBUG" in os.environ:
    print("디버그 모드가 설정되어 있습니다")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  환경변수 타입 변환 헬퍼 함수               │
# │  os.getenv()는 항상 문자열을 반환            │
# │  정수, 불리언, 리스트로 변환하는 함수 작성    │
# └─────────────────────────────────────────┘

import os

def get_env_int(key, default=0):
    """환경변수를 정수로 반환"""
    value = os.getenv(key)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        return default

def get_env_bool(key, default=False):
    """환경변수를 불리언으로 반환"""
    value = os.getenv(key)
    if value is None:
        return default
    return value.lower() in ("true", "1", "yes", "on")

def get_env_list(key, separator=",", default=None):
    """환경변수를 리스트로 반환"""
    value = os.getenv(key)
    if value is None:
        return default or []
    return value.split(separator)

# 테스트
os.environ["MY_PORT"] = "9090"
os.environ["MY_DEBUG"] = "true"
os.environ["MY_HOSTS"] = "host1,host2,host3"

print(f"포트 (int): {get_env_int('MY_PORT')}")
print(f"디버그 (bool): {get_env_bool('MY_DEBUG')}")
print(f"호스트 (list): {get_env_list('MY_HOSTS')}")
print(f"기본값 테스트: {get_env_int('MISSING', 42)}")

## 2. configparser — INI 설정 파일

INI 형식의 설정 파일을 읽고 쓰는 표준 라이브러리입니다.
섹션(`[section]`)과 키-값 쌍으로 구성됩니다.

In [ ]:
import configparser, os, tempfile

tmp = tempfile.gettempdir()
config_path = os.path.join(tmp, "tutorial_config.ini")

# ┌─────────────────────────────────────────┐
# │  INI 파일 형식                              │
# │  [section]                                 │
# │  key = value                               │
# │  key2 = value2                             │
# └─────────────────────────────────────────┘

# 설정 파일 생성
config = configparser.ConfigParser()

# 섹션과 키-값 쌍 추가
config["database"] = {
    "host": "localhost",
    "port": "5432",
    "name": "myapp_db",
    "user": "admin",
}

config["server"] = {
    "host": "0.0.0.0",
    "port": "8080",
    "debug": "false",
}

# 파일에 쓰기
with open(config_path, "w") as f:
    config.write(f)

print("설정 파일 생성 완료")

In [ ]:
# 설정 파일 읽기
import configparser, os, tempfile

tmp = tempfile.gettempdir()
config_path = os.path.join(tmp, "tutorial_config.ini")

config = configparser.ConfigParser()
config.read(config_path, encoding="utf-8")

# 섹션 목록
print(f"섹션: {config.sections()}")

# 특정 섹션의 키-값 읽기
db_host = config["database"]["host"]
db_port = config["database"]["port"]
print(f"DB 호스트: {db_host}")
print(f"DB 포트: {db_port}")

# 타입 변환
db_port_int = config.getint("database", "port")
srv_debug = config.getboolean("server", "debug")
print(f"DB 포트 (int): {db_port_int}")
print(f"디버그 (bool): {srv_debug}")

# 기본값 설정
max_conn = config.get("database", "max_connections", fallback="100")
print(f"최대 연결: {max_conn}")

# 전체 섹션 출력
for section in config.sections():
    print(f"\n[{section}]")
    for key, value in config[section].items():
        print(f"  {key} = {value}")

## 🎯 연습 문제

1. `os.getenv()`를 사용해 `HOME`(또는 `USERPROFILE`)과 `PATH` 환경변수를 출력하세요.
2. 환경변수 `APP_MODE`가 `production`이면 `"프로덕션 모드"`, 아니면 `"개발 모드"`를 출력하는 코드를 작성하세요.
3. INI 파일에 `[logging]` 섹션을 추가하고 `level=INFO`, `file=app.log`를 설정하세요.
4. `get_env_float()` 헬퍼 함수를 작성하세요 (소수점 환경변수 반환, 기본값 0.0).